In [ ]:
import numpy as np
import cv2
import collections
import keras
from keras import layers, models, Sequential, optimizers, losses
import random as rand
import tensorflow as tf
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# --- Configuration Flags ---
TRAINING = True
RENDERING = False   # Set to True to see the game (requires resources)
EPISODES = 3000 # Hack after 6000
MAX_STEPS = 10000
LOADING = True
SAVING = True
SAVE_EVERY = 100
FRAME_SKIP = 4      # Action hold (frame skip) count
SCREEN_HEIGHT = 800
SCREEN_WIDTH = 850
CHECKPOINT_DIR = '/content/drive/MyDrive/SpaceInvadersRL/'

# Preprocessed image dimensions (downscaled for efficiency)
INPUT_HEIGHT = 84
INPUT_WIDTH = 84
INPUT_CHANNELS = 3

class Agent:
    """DQN Agent for Space Invaders with epsilon-greedy exploration and target network."""

    def __init__(self, gamma=0.99, epsilon=0.2, epsilon_min=0.05, epsilon_decay=0.9995,
                 learning_rate=0.00007, target_update_freq=1000):
        self.replay_memory = collections.deque(maxlen=50000)
        self.batch_size = 32
        self.gamma = gamma

        # Exploration parameters
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay

        # Target network update frequency
        self.target_update_freq = target_update_freq
        self.train_step_counter = 0

        # Build main model (for action selection and training)
        self.model = self._build_model()
        self.model.compile(
            optimizer=optimizers.Adam(learning_rate=learning_rate),
            loss=losses.MeanSquaredError()
        )

        # Build target model (for stable Q-value targets)
        self.target_model = self._build_model()
        self.update_target_network()

    def _build_model(self):
        """Build a CNN architecture suitable for the input size."""
        return Sequential([
            layers.Input(shape=(INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS)),
            layers.Conv2D(32, kernel_size=(8, 8), strides=4, padding='same'),
            layers.ReLU(),
            layers.Conv2D(64, kernel_size=(4, 4), strides=2, padding='same'),
            layers.ReLU(),
            layers.Conv2D(64, kernel_size=(3, 3), strides=1, padding='same'),
            layers.ReLU(),
            layers.Flatten(),
            layers.Dense(512),
            layers.ReLU(),
            layers.Dense(4)  # 4 actions: no move, left, right, shoot
        ])

    def update_target_network(self):
        """Copy weights from main model to target model."""
        self.target_model.set_weights(self.model.get_weights())

    def load_model_weights(self, path):
        """Load weights into both main and target models."""
        self.model.load_weights(path)
        self.update_target_network()

    def save_model_weights(self, path):
        self.model.save_weights(path)

    def select_action(self, state, training=True):
        """Select an action using epsilon-greedy policy."""
        if len(state.shape) == 3:
            state = np.expand_dims(state, axis=0)

        # Epsilon-greedy exploration
        if training and rand.random() < self.epsilon:
            return rand.randint(0, 3)  # Random action

        # Optimized inference using __call__ instead of predict
        prediction = self.model(state, training=False).numpy()
        return np.argmax(prediction)

    def decay_epsilon(self):
        """Decay epsilon after each episode."""
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def store_experience(self, state, action, reward, newstate, done, display=False):
        """Store the experience tuple in replay memory."""
        if display:
            cv2.imshow("State Image", state)
            cv2.waitKey(1)
        self.replay_memory.append((state, action, reward, newstate, done))

    def train_model(self):
        """Train the model using experience replay with batch updates."""
        if len(self.replay_memory) < self.batch_size:
            return

        # Sample a batch
        batch = rand.sample(self.replay_memory, self.batch_size)

        # Prepare batch arrays
        states = np.array([exp[0] for exp in batch])
        actions = np.array([exp[1] for exp in batch])
        rewards = np.array([exp[2] for exp in batch])
        next_states = np.array([exp[3] for exp in batch])
        dones = np.array([exp[4] for exp in batch])

        # Get Q-values (optimized inference)
        current_q_values = self.model(states, training=False).numpy()
        next_q_values = self.target_model(next_states, training=False).numpy()

        # Compute target Q-values using Bellman equation
        targets = current_q_values.copy()
        batch_indices = np.arange(self.batch_size)

        # Vectorized target update
        targets[batch_indices, actions] = rewards + (1 - dones) * self.gamma * np.max(next_q_values, axis=1)

        # Train on the batch (faster than fit)
        self.model.train_on_batch(states, targets)

        # Increment step counter and update target network periodically
        self.train_step_counter += 1
        if self.train_step_counter % self.target_update_freq == 0:
            self.update_target_network()

Mounted at /content/drive


In [ ]:
"""Space Invaders Reinforcement Learning Environment.

This module implements a Space Invaders game environment for training RL agents.
The environment follows a gym-like interface with step() and reset() methods.
"""

import sys
from typing import Tuple, List, Optional

class NullWriter:
    """A writer that suppresses pygame initialization output."""
    def write(self, arg: str) -> None: pass
    def flush(self) -> None: pass

original = sys.stdout
sys.stdout=NullWriter()

#Import Modules
import pygame
import random as rand
import numpy as np

sys.stdout = original

class Environment():
    """Space Invaders game environment for reinforcement learning.

    Optimized with vectorized enemy management.
    """

    pygame_initialized: bool = False

    @classmethod
    def _initialize_pygame(cls) -> None:
        """Initialize pygame if not already initialized."""
        pygame.init()
        cls.pygame_initialized = True

    def __init__(self, ammo_inc: float, Player_Speed: float, Enemy_Speed: float,
                 starting_ammo: float, num_enem: int, ammo_penalty: float,
                 hit_reward: float, death_penalty: float, closeness_penalty: float,
                 closeness_threshold: float, SCREEN_HEIGHT: int = 800, SCREEN_WIDTH: int = 850) -> None:
        self.rendering: bool = False
        self.ammo_inc = ammo_inc
        self.Player_Speed = Player_Speed
        self.Enemy_Speed = Enemy_Speed
        self.starting_ammo = starting_ammo
        self.num_enem = num_enem
        self.ammo_penalty = ammo_penalty
        self.hit_reward = hit_reward
        self.death_penalty = death_penalty
        self.closeness_penalty = closeness_penalty
        self.closeness_threshold = closeness_threshold
        self.SCREEN_HEIGHT=SCREEN_HEIGHT
        self.SCREEN_WIDTH=SCREEN_WIDTH

        self._initialize_pygame()
        self.reset()
        self.total_reward: float = 0

    def initialize_rendering(self) -> None:
        self.rendering = True
        self.font = pygame.font.Font("resources/TNR.ttf", 32)
        self.gofont = pygame.font.Font("resources/TNR.ttf", 100)
        self.scfont = pygame.font.Font("resources/TNR.ttf", 80)
        self.screen = pygame.display.set_mode((self.SCREEN_WIDTH, self.SCREEN_HEIGHT))
        self.Background = pygame.image.load("resources/space.jpg")
        pygame.display.set_caption("Space Invaders")
        icon = pygame.image.load("resources/ufo.png")
        pygame.display.set_icon(icon)
        self.PlayerImg = pygame.image.load("resources/hero.png")
        self.bulletImg = pygame.image.load("resources/bolt.png")
        self.enemyImg = pygame.image.load("resources/ufo.png")

        self.player = lambda x, y: self.screen.blit(self.PlayerImg, (x, y))
        self.textX = 10
        self.textY = 10

    def show_score(self, x: int, y: int) -> None:
        score = self.font.render("Score: %d" %self.score_value, True, (255, 255, 255))
        self.screen.blit(score, (x, y))

    def show_ammo(self, x: int, y: int) -> None:
        ammo = self.font.render("Ammo: %.1f" %self.ammo_value, True, (255, 255, 255))
        self.screen.blit(ammo, (x, y))

    def reset(self) -> None:
        self.game_over = False
        self.playerX = 400.0
        self.playerY = 730.0
        self.playerXD = 0.0

        self.bulletX = 400.0
        self.bulletY = 730.0
        self.bulletXD = 0.0
        self.bulletYD = 20.0
        self.bullet_state = "ready"
        self.ammo_value = self.starting_ammo

        self.score_value = 0
        self.reward = 0

        # --- Vectorized Enemy State ---
        # Positions
        self.enemies_x = np.random.randint(0, 750, self.num_enem).astype(np.float32)
        self.enemies_y = np.random.randint(50, 150, self.num_enem).astype(np.float32)
        # Velocity X: Randomly -Speed or +Speed
        self.enemies_dx = (np.random.randint(0, 2, self.num_enem) * 2 - 1) * self.Enemy_Speed
        self.enemies_dx = self.enemies_dx.astype(np.float32)
        # Velocity Y (drop amount)
        self.enemies_yd = 40.0

        def fire_bullet(x: float, y: float) -> None:
            self.bullet_state = "fire"
            if self.rendering:
                self.screen.blit(self.bulletImg, (x, y))
        self.fire_bullet = fire_bullet

    def move_player(self) -> None:
        self.playerX += self.playerXD
        if self.playerX < 0: self.playerX = 0
        elif self.playerX > 786: self.playerX = 786

    def move_bullet(self) -> None:
        if self.bullet_state == "fire":
            if self.rendering:
                self.screen.blit(self.bulletImg, (self.bulletX, self.bulletY))
            self.bulletY -= self.bulletYD
            if self.bulletY < 0:
                self.bulletY = 730
                self.bullet_state = "ready"
                if self.ammo_value < 1:
                    self.game_over = True

    def state(self) -> np.ndarray:
        """Get the current state as a numpy array. Format matches previous implementation for compatibility."""
        # Flatten enemies: (x, y, dx) interleaved
        enemies_flat = np.column_stack((self.enemies_x, self.enemies_y, self.enemies_dx)).flatten()

        res = np.concatenate(([self.playerX, self.playerXD,
                               1.0 if self.bullet_state == "ready" else 0.0,
                               self.bulletX if self.bullet_state=="fire" else 0.0,
                               self.bulletY if self.bullet_state =="fire" else 0.0],
                              enemies_flat))
        return res.reshape(1, 1, -1)

    def step(self, action: int) -> Tuple[float, np.ndarray, np.ndarray, bool]:
        state = self.state()

        # Action Logic
        if action == 0: self.playerXD = 0
        elif action == 1: self.playerXD = -self.Player_Speed
        elif action == 2: self.playerXD = self.Player_Speed
        elif action == 3 and self.bullet_state == "ready":
            self.bulletX = self.playerX
            self.bullet_state = "fire"
            self.ammo_value -= 1
            self.reward -= self.ammo_penalty
            self.total_reward -= self.ammo_penalty

        if self.ammo_value < 1 and self.bullet_state == "ready":
            self.game_over = True

        # --- Vectorized Enemy Logic ---
        self.enemies_x += self.enemies_dx

        # Wall Bouncing (Right)
        right_mask = self.enemies_x > 786
        self.enemies_dx[right_mask] = -self.Enemy_Speed
        self.enemies_y[right_mask] += self.enemies_yd

        # Wall Bouncing (Left)
        left_mask = self.enemies_x < 0
        self.enemies_dx[left_mask] = self.Enemy_Speed
        self.enemies_y[left_mask] += self.enemies_yd

        # Collision: Bullet vs Enemies
        if self.bullet_state == "fire":
            # Calculate squared distances (faster than sqrt)
            dist_sq = (self.enemies_x - self.bulletX)**2 + (self.enemies_y - self.bulletY)**2
            # Threshold 27 -> 27^2 = 729
            hits = dist_sq < 27**2

            if np.any(hits):
                n_hits = np.sum(hits)
                self.reward += self.hit_reward * n_hits
                self.total_reward += self.hit_reward * n_hits
                self.score_value += n_hits
                self.ammo_value += self.ammo_inc * n_hits

                # Reset bullet
                self.bullet_state = "ready"
                self.bulletY = 730

                # Respawn hit enemies
                self.enemies_x[hits] = np.random.randint(0, 750, n_hits)
                self.enemies_y[hits] = np.random.randint(50, 150, n_hits)
                self.enemies_dx[hits] = (np.random.randint(0, 2, n_hits) * 2 - 1) * self.Enemy_Speed

        # Collision: Player vs Enemies
        if not self.game_over:
            dist_sq_p = (self.enemies_x - self.playerX)**2 + (self.enemies_y - self.playerY)**2
            # Threshold 33 -> 33^2 = 1089
            if np.any(dist_sq_p < 33**2):
                self.game_over = True

        # Phobia Penalty
        threshold_y = self.SCREEN_HEIGHT * self.closeness_threshold
        phobia_mask = self.enemies_y > threshold_y
        if np.any(phobia_mask):
             penalties = (self.enemies_y[phobia_mask] / self.SCREEN_HEIGHT) * self.closeness_penalty
             total_pen = np.sum(penalties)
             self.reward -= total_pen
             self.total_reward -= total_pen

        self.move_player()
        self.move_bullet()

        newstate = self.state()

        if self.game_over:
            self.reward -= self.death_penalty
            self.total_reward -= self.death_penalty

        reward = float(self.reward)
        self.reward = 0

        return (reward, state, newstate, self.game_over)

    def render(self) -> None:
        if not self.rendering:
            return
        self.screen.fill((0,0,0))
        self.screen.blit(self.Background, (0,0))

        # Loop over arrays to render
        for i in range(self.num_enem):
            self.screen.blit(self.enemyImg, (self.enemies_x[i], self.enemies_y[i]))

        self.player(self.playerX, self.playerY)
        if not self.game_over:
            self.show_score(self.textX, self.textY)
            self.show_ammo(8,40)
        pygame.display.flip()

    def get_picture_as_numpy(self) -> np.ndarray:
        if not self.rendering:
            raise RuntimeError("Rendering not initialized. Call initialize_rendering() first.")
        return np.array(pygame.surfarray.pixels3d(self.screen))

In [ ]:
import numpy as np
import cv2

# Target dimensions for the neural network (much more efficient)
TARGET_HEIGHT = 84
TARGET_WIDTH = 84

class Preprocessor:
    def __init__(self, screen_height, screen_width):
        self.screen_height = screen_height
        self.screen_width = screen_width
        self.target_height = TARGET_HEIGHT
        self.target_width = TARGET_WIDTH

    def preprocess(self, positions: np.ndarray) -> np.ndarray:
        """
        Turn the game state into a simple image (BGR format for OpenCV), then downscale.
        Enemies at their positions are red if going right, blue if going left.
        Player is green if bullet is ready, yellow if bullet is fired.

        Args:
            positions (np.ndarray): Current game state positions from environment (shape: (1, 1, 23)).

        Returns:
            Downscaled image of shape (84, 84, 3) normalized to [0, 1].
        """

        if isinstance(positions, np.ndarray):
            positions = positions.flatten()
        elif isinstance(positions, tuple):
            pass
        else:
            raise TypeError("Ummm. What's that?")

        image = np.zeros((self.screen_height, self.screen_width, 3), dtype=np.uint8)

        # Unpack positions
        playerX, playerXD, bullet_ready, bulletX, bulletY = positions[:5]
        enemy_positions = positions[5:]

        # Draw player
        player_color = (0, 255, 0) if bullet_ready == 1 else (0, 255, 255)
        image[750:770, int(playerX):int(playerX)+50] = player_color

        # Draw bullet
        if bullet_ready == 0:
            image[int(bulletY):int(bulletY)+10, int(bulletX)+24:int(bulletX)+26] = (255, 255, 255)

        # Draw enemies
        for i in range(0, len(enemy_positions), 3):
            enemyX = enemy_positions[i]
            enemyY = enemy_positions[i+1]
            enemyXD = enemy_positions[i+2]
            enemy_color = (0, 0, 255) if enemyXD > 0 else (255, 0, 0)
            image[int(enemyY):int(enemyY)+40, int(enemyX):int(enemyX)+40] = enemy_color

        # Downscale image for neural network efficiency
        image = cv2.resize(image, (self.target_width, self.target_height), interpolation=cv2.INTER_AREA)

        # Normalize to [0, 1] for better neural network training
        image = image.astype(np.float32) / 255.0

        return image

In [ ]:
import tempfile
import shutil
import os

"""
Deep Space - DQN Agent for Space Invaders
==========================================
Training Configuration:
- Set TRAINING=True, RENDERING=False for fast training
- Set TRAINING=False, RENDERING=True to watch trained agent
- LOADING=True loads previous weights (won't work with new architecture)
"""

from tqdm import tqdm
import sys
import pygame
import numpy as np
import cv2
import random as rand
from typing import Tuple, List, Optional

# --- Definitions to fix NameError if previous cells weren't run ---

class NullWriter:
    """A writer that suppresses pygame initialization output."""
    def write(self, arg: str) -> None: pass
    def flush(self) -> None: pass

# Redirect stdout to suppress pygame welcome message
original_stdout = sys.stdout
sys.stdout = NullWriter()
import pygame
sys.stdout = original_stdout

class Environment():
    """Space Invaders game environment for reinforcement learning.
    Optimized with vectorized enemy management.
    """

    pygame_initialized: bool = False

    @classmethod
    def _initialize_pygame(cls) -> None:
        """Initialize pygame if not already initialized."""
        pygame.init()
        cls.pygame_initialized = True

    def __init__(self, ammo_inc: float, Player_Speed: float, Enemy_Speed: float,
                 starting_ammo: float, num_enem: int, ammo_penalty: float,
                 hit_reward: float, death_penalty: float, closeness_penalty: float,
                 closeness_threshold: float, SCREEN_HEIGHT: int = 800, SCREEN_WIDTH: int = 850) -> None:
        self.rendering: bool = False
        self.ammo_inc = ammo_inc
        self.Player_Speed = Player_Speed
        self.Enemy_Speed = Enemy_Speed
        self.starting_ammo = starting_ammo
        self.num_enem = num_enem
        self.ammo_penalty = ammo_penalty
        self.hit_reward = hit_reward
        self.death_penalty = death_penalty
        self.closeness_penalty = closeness_penalty
        self.closeness_threshold = closeness_threshold
        self.SCREEN_HEIGHT=SCREEN_HEIGHT
        self.SCREEN_WIDTH=SCREEN_WIDTH

        self._initialize_pygame()
        self.reset()
        self.total_reward: float = 0

    def initialize_rendering(self) -> None:
        self.rendering = True
        self.font = pygame.font.Font("resources/TNR.ttf", 32)
        self.gofont = pygame.font.Font("resources/TNR.ttf", 100)
        self.scfont = pygame.font.Font("resources/TNR.ttf", 80)
        self.screen = pygame.display.set_mode((self.SCREEN_WIDTH, self.SCREEN_HEIGHT))
        self.Background = pygame.image.load("resources/space.jpg")
        pygame.display.set_caption("Space Invaders")
        icon = pygame.image.load("resources/ufo.png")
        pygame.display.set_icon(icon)
        self.PlayerImg = pygame.image.load("resources/hero.png")
        self.bulletImg = pygame.image.load("resources/bolt.png")
        self.enemyImg = pygame.image.load("resources/ufo.png")

        self.player = lambda x, y: self.screen.blit(self.PlayerImg, (x, y))
        self.textX = 10
        self.textY = 10

    def show_score(self, x: int, y: int) -> None:
        score = self.font.render("Score: %d" %self.score_value, True, (255, 255, 255))
        self.screen.blit(score, (x, y))

    def show_ammo(self, x: int, y: int) -> None:
        ammo = self.font.render("Ammo: %.1f" %self.ammo_value, True, (255, 255, 255))
        self.screen.blit(ammo, (x, y))

    def reset(self) -> None:
        self.game_over = False
        self.playerX = 400.0
        self.playerY = 730.0
        self.playerXD = 0.0

        self.bulletX = 400.0
        self.bulletY = 730.0
        self.bulletXD = 0.0
        self.bulletYD = 20.0
        self.bullet_state = "ready"
        self.ammo_value = self.starting_ammo

        self.score_value = 0
        self.reward = 0

        # --- Vectorized Enemy State ---
        # Positions
        self.enemies_x = np.random.randint(0, 750, self.num_enem).astype(np.float32)
        self.enemies_y = np.random.randint(50, 150, self.num_enem).astype(np.float32)
        # Velocity X: Randomly -Speed or +Speed
        self.enemies_dx = (np.random.randint(0, 2, self.num_enem) * 2 - 1) * self.Enemy_Speed
        self.enemies_dx = self.enemies_dx.astype(np.float32)
        # Velocity Y (drop amount)
        self.enemies_yd = 40.0

        def fire_bullet(x: float, y: float) -> None:
            self.bullet_state = "fire"
            if self.rendering:
                self.screen.blit(self.bulletImg, (x, y))
        self.fire_bullet = fire_bullet

    def move_player(self) -> None:
        self.playerX += self.playerXD
        if self.playerX < 0: self.playerX = 0
        elif self.playerX > 786: self.playerX = 786

    def move_bullet(self) -> None:
        if self.bullet_state == "fire":
            if self.rendering:
                self.screen.blit(self.bulletImg, (self.bulletX, self.bulletY))
            self.bulletY -= self.bulletYD
            if self.bulletY < 0:
                self.bulletY = 730
                self.bullet_state = "ready"
                if self.ammo_value < 1:
                    self.game_over = True

    def state(self) -> np.ndarray:
        """Get the current state as a numpy array. Format matches previous implementation for compatibility."""
        # Flatten enemies: (x, y, dx) interleaved
        enemies_flat = np.column_stack((self.enemies_x, self.enemies_y, self.enemies_dx)).flatten()

        res = np.concatenate(([self.playerX, self.playerXD,
                               1.0 if self.bullet_state == "ready" else 0.0,
                               self.bulletX if self.bullet_state=="fire" else 0.0,
                               self.bulletY if self.bullet_state =="fire" else 0.0],
                              enemies_flat))
        return res.reshape(1, 1, -1)

    def step(self, action: int) -> Tuple[float, np.ndarray, np.ndarray, bool]:
        state = self.state()

        # Action Logic
        if action == 0: self.playerXD = 0
        elif action == 1: self.playerXD = -self.Player_Speed
        elif action == 2: self.playerXD = self.Player_Speed
        elif action == 3 and self.bullet_state == "ready":
            self.bulletX = self.playerX
            self.bullet_state = "fire"
            self.ammo_value -= 1
            self.reward -= self.ammo_penalty
            self.total_reward -= self.ammo_penalty

        if self.ammo_value < 1 and self.bullet_state == "ready":
            self.game_over = True

        # --- Vectorized Enemy Logic ---
        self.enemies_x += self.enemies_dx

        # Wall Bouncing (Right)
        right_mask = self.enemies_x > 786
        self.enemies_dx[right_mask] = -self.Enemy_Speed
        self.enemies_y[right_mask] += self.enemies_yd

        # Wall Bouncing (Left)
        left_mask = self.enemies_x < 0
        self.enemies_dx[left_mask] = self.Enemy_Speed
        self.enemies_y[left_mask] += self.enemies_yd

        # Collision: Bullet vs Enemies
        if self.bullet_state == "fire":
            # Calculate squared distances (faster than sqrt)
            dist_sq = (self.enemies_x - self.bulletX)**2 + (self.enemies_y - self.bulletY)**2
            # Threshold 27 -> 27^2 = 729
            hits = dist_sq < 27**2

            if np.any(hits):
                n_hits = np.sum(hits)
                self.reward += self.hit_reward * n_hits
                self.total_reward += self.hit_reward * n_hits
                self.score_value += n_hits
                self.ammo_value += self.ammo_inc * n_hits

                # Reset bullet
                self.bullet_state = "ready"
                self.bulletY = 730

                # Respawn hit enemies
                self.enemies_x[hits] = np.random.randint(0, 750, n_hits)
                self.enemies_y[hits] = np.random.randint(50, 150, n_hits)
                self.enemies_dx[hits] = (np.random.randint(0, 2, n_hits) * 2 - 1) * self.Enemy_Speed

        # Collision: Player vs Enemies
        if not self.game_over:
            dist_sq_p = (self.enemies_x - self.playerX)**2 + (self.enemies_y - self.playerY)**2
            # Threshold 33 -> 33^2 = 1089
            if np.any(dist_sq_p < 33**2):
                self.game_over = True

        # Phobia Penalty
        threshold_y = self.SCREEN_HEIGHT * self.closeness_threshold
        phobia_mask = self.enemies_y > threshold_y
        if np.any(phobia_mask):
             penalties = (self.enemies_y[phobia_mask] / self.SCREEN_HEIGHT) * self.closeness_penalty
             total_pen = np.sum(penalties)
             self.reward -= total_pen
             self.total_reward -= total_pen

        self.move_player()
        self.move_bullet()

        newstate = self.state()

        if self.game_over:
            self.reward -= self.death_penalty
            self.total_reward -= self.death_penalty

        reward = float(self.reward)
        self.reward = 0

        return (reward, state, newstate, self.game_over)

    def render(self) -> None:
        if not self.rendering:
            return
        self.screen.fill((0,0,0))
        self.screen.blit(self.Background, (0,0))

        # Loop over arrays to render
        for i in range(self.num_enem):
            self.screen.blit(self.enemyImg, (self.enemies_x[i], self.enemies_y[i]))

        self.player(self.playerX, self.playerY)
        if not self.game_over:
            self.show_score(self.textX, self.textY)
            self.show_ammo(8,40)
        pygame.display.flip()

    def get_picture_as_numpy(self) -> np.ndarray:
        if not self.rendering:
            raise RuntimeError("Rendering not initialized. Call initialize_rendering() first.")
        return np.array(pygame.surfarray.pixels3d(self.screen))

# Target dimensions
TARGET_HEIGHT = 84
TARGET_WIDTH = 84

class Preprocessor:
    def __init__(self, screen_height, screen_width):
        self.screen_height = screen_height
        self.screen_width = screen_width
        self.target_height = TARGET_HEIGHT
        self.target_width = TARGET_WIDTH

    def preprocess(self, positions: np.ndarray) -> np.ndarray:
        """
        Turn the game state into a simple image (BGR format for OpenCV), then downscale.
        Enemies at their positions are red if going right, blue if going left.
        Player is green if bullet is ready, yellow if bullet is fired.
        """

        if isinstance(positions, np.ndarray):
            positions = positions.flatten()
        elif isinstance(positions, tuple):
            pass
        else:
            raise TypeError("Ummm. What's that?")

        image = np.zeros((self.screen_height, self.screen_width, 3), dtype=np.uint8)

        # Unpack positions
        playerX, playerXD, bullet_ready, bulletX, bulletY = positions[:5]
        enemy_positions = positions[5:]

        # Draw player
        player_color = (0, 255, 0) if bullet_ready == 1 else (0, 255, 255)
        image[750:770, int(playerX):int(playerX)+50] = player_color

        # Draw bullet
        if bullet_ready == 0:
            image[int(bulletY):int(bulletY)+10, int(bulletX)+24:int(bulletX)+26] = (255, 255, 255)

        # Draw enemies
        for i in range(0, len(enemy_positions), 3):
            enemyX = enemy_positions[i]
            enemyY = enemy_positions[i+1]
            enemyXD = enemy_positions[i+2]
            enemy_color = (0, 0, 255) if enemyXD > 0 else (255, 0, 0)
            image[int(enemyY):int(enemyY)+40, int(enemyX):int(enemyX)+40] = enemy_color

        # Downscale image for neural network efficiency
        image = cv2.resize(image, (self.target_width, self.target_height), interpolation=cv2.INTER_AREA)

        # Normalize to [0, 1] for better neural network training
        image = image.astype(np.float32) / 255.0

        return image

# --- Main Training Script ---
# NOTE: Configuration flags (TRAINING, RENDERING, etc.) are defined in the previous cell.

# Environment with tuned reward parameters
env = Environment(
    ammo_inc=1.5,           # Ammo gained per hit
    Player_Speed=2,         # Faster player movement
    Enemy_Speed=1,
    starting_ammo=10,
    num_enem=6,
    ammo_penalty=0.67,      # Small penalty for shooting (was 1). This is a hack because it used to be .35, which resulted in the trigger happy behavior we see today. Try the whole thing again but get the rewards to reflect a little more conservative of a strategy than is required by ammo laws. You can have one miss for every two hits, so (3*fire penalty) + (2*hit_reward) = 0, so hit reward = 1; fire penalty = 0.67? Yes. So we could do that from the start. Myabe make it hit reward = 1; fire penalty = 0.7.
    hit_reward=1.0,         # Big reward for hitting (was 1.5)
    death_penalty=0.0,      # Big penalty for dying (was 100)
    closeness_penalty=0.1,  # Reduced (was 0.5)
    closeness_threshold=0.5,
    SCREEN_HEIGHT=SCREEN_HEIGHT,
    SCREEN_WIDTH=SCREEN_WIDTH
)

preprocessor = Preprocessor(SCREEN_HEIGHT, SCREEN_WIDTH)

agent = Agent(
    gamma=0.99,
    epsilon=0.2 if TRAINING else 0.0,  # No exploration when testing
    epsilon_min=0.05,
    epsilon_decay=0.9995,
    learning_rate=0.00007,#Hack
    target_update_freq=1000
)

# --- Patch agent's save_model_weights method to handle Drive write issues ---
def patched_save_model_weights(self, path):
    # Fix: Keras requires the filename to end in .weights.h5 for save_weights
    with tempfile.NamedTemporaryFile(suffix=".weights.h5", delete=False) as tmp_file:
        temp_path = tmp_file.name
    try:
        self.model.save_weights(temp_path)
        shutil.copy(temp_path, path)
        print(f"Successfully saved weights to {path} via temporary file.")
    except Exception as e:
        print(f"Error saving weights to {path} even with temp file: {e}")
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)

# Apply the patch to the agent instance
# We need to access the Agent class from where it was defined (likely cell 5a5b11fa)
# Assuming Agent class is globally available, this will work. If not, will need to import.
# Since it's available in the kernel state, it should be fine.
agent.save_model_weights = patched_save_model_weights.__get__(agent, type(agent))

# --- Drive Saving Setup ---
model_path = os.path.join(CHECKPOINT_DIR, "model.weights.h5")
best_model_path = os.path.join(CHECKPOINT_DIR, "model_best.weights.h5")
steps_path = os.path.join(CHECKPOINT_DIR, "steps.txt")

if SAVING:
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"Saving checkpoints to: {CHECKPOINT_DIR}")

start_episode = 0

if LOADING:
    # Load steps if exists
    if os.path.exists(steps_path):
        try:
            with open(steps_path, 'r') as f:
                content = f.read().strip()
                if content:
                    start_episode = int(content)
                    start_episode = 0 #Override with a hack
            print(f"Loaded start episode: {start_episode}")

            # Decay epsilon
            if TRAINING and start_episode > 0:
                new_epsilon = agent.epsilon * (agent.epsilon_decay ** start_episode)
                agent.epsilon = max(agent.epsilon_min, new_epsilon)
                print(f"Adjusted epsilon to: {agent.epsilon:.4f}")

        except Exception as e:
            print(f"Error loading steps.txt: {e}")


    try:
        agent.load_model_weights(model_path)
        print(f"Loaded model weights successfully from {model_path}!")
    except Exception as e:
        print(f"Could not load weights from {model_path}: {e}")
        # Fallback to local file
        local_weights = "model.weights.h5"
        if os.path.exists(local_weights):
            print(f"Falling back to local weights: {local_weights}")
            try:
                agent.load_model_weights(local_weights)
                print("Loaded local weights successfully!")
            except Exception as e_local:
                print(f"Could not load local weights: {e_local}")
                print("Starting with fresh weights.")
        else:
            print(f"Local weights file {local_weights} not found. Starting with fresh weights.")

if RENDERING:
    env.initialize_rendering()

# Training metrics
episode_rewards = []
best_reward = float('-inf')

# Adjusted loop range
for episode in tqdm(range(start_episode, EPISODES), desc="Training", initial=start_episode, total=EPISODES):
    env.reset()
    state = env.state()
    processed_state = preprocessor.preprocess(state)

    episode_reward = 0

    for step in range(MAX_STEPS):
        action = agent.select_action(processed_state, training=TRAINING)

        # Frame Skipping Logic
        current_reward = 0
        for _ in range(FRAME_SKIP):
            reward, _, newstate, done = env.step(action)
            current_reward += reward

            if RENDERING:
                # Process pygame events to keep window responsive
                for event in pygame.event.get():
                    if event.type == pygame.QUIT:
                        pygame.quit()
                        sys.exit(0)
                env.render()

            if done:
                break

        newstate_processed = preprocessor.preprocess(newstate)
        episode_reward += current_reward

        if TRAINING:
            agent.store_experience(processed_state, action, current_reward, newstate_processed, done)
            agent.train_model()

        processed_state = newstate_processed

        if done:
            break

    # Decay epsilon after each episode
    if TRAINING:
        agent.decay_epsilon()

    episode_rewards.append(episode_reward)

    # Print progress every 50 episodes
    if (episode + 1) % 50 == 0:
        avg_reward = np.mean(episode_rewards[-50:])
        print(f"\nEpisode {episode+1} | Avg Reward (last 50): {avg_reward:.2f} | Epsilon: {agent.epsilon:.3f}")

    # Save best model and periodic saves and make sure we always keep the best one
    if SAVING:
        # Save steps helper
        def save_steps(step_count):
            try:
                with open(steps_path, 'w') as f:
                    f.write(str(step_count))
            except Exception as e:
                print(f"Error saving steps: {e}")

        if episode_reward > best_reward:
            best_reward = episode_reward
            agent.save_model_weights(best_model_path)
            save_steps(episode + 1)

        if (episode + 1) % SAVE_EVERY == 0:
            agent.save_model_weights(model_path)
            save_steps(episode + 1)
            print(f"\nSaved checkpoint at episode {episode+1}")

if SAVING:
    agent.save_model_weights(model_path)
    # Save final steps
    with open(steps_path, 'w') as f:
        f.write(str(EPISODES))
    print("\nTraining complete! Model saved.")

Saving checkpoints to: /content/drive/MyDrive/SpaceInvadersRL/
Loaded start episode: 0


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Loaded model weights successfully from /content/drive/MyDrive/SpaceInvadersRL/model.weights.h5!


Training:   0%|          | 1/3000 [00:17<14:34:29, 17.50s/it]

Successfully saved weights to /content/drive/MyDrive/SpaceInvadersRL/model_best.weights.h5 via temporary file.


...

Training:  83%|████████▎ | 2500/3000 [12:02:10<2:36:47, 18.82s/it]

Successfully saved weights to /content/drive/MyDrive/SpaceInvadersRL/model.weights.h5 via temporary file.

Saved checkpoint at episode 2500


Training:  85%|████████▌ | 2550/3000 [12:23:32<4:48:31, 38.47s/it]


Episode 2550 | Avg Reward (last 50): -7.65 | Epsilon: 0.056


Training:  86%|████████▌ | 2577/3000 [12:36:48<3:37:30, 30.85s/it]